In [ ]:
import pandas as pd
import re
import json, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, EsmForMaskedLM
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from Bio.Align import substitution_matrices


In [2]:
df = pd.read_csv("variant_summary.txt.gz", sep="\t", low_memory=False)
print(df.shape)
print(df.columns.tolist())
print(df.head(3).T)  

# Check the exact strings used in each filtered column
for col in ["Assembly", "Type", "GermlineClassification", "MolecularConsequence", "ReviewStatus"]:
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].value_counts(dropna=False).head(15))

(9311986, 43)
['#AlleleID', 'Type', 'Name', 'GeneID', 'GeneSymbol', 'HGNC_ID', 'ClinicalSignificance', 'ClinSigSimple', 'LastEvaluated', 'RS# (dbSNP)', 'nsv/esv (dbVar)', 'RCVaccession', 'PhenotypeIDS', 'PhenotypeList', 'Origin', 'OriginSimple', 'Assembly', 'ChromosomeAccession', 'Chromosome', 'Start', 'Stop', 'ReferenceAllele', 'AlternateAllele', 'Cytogenetic', 'ReviewStatus', 'NumberSubmitters', 'Guidelines', 'TestedInGTR', 'OtherIDs', 'SubmitterCategories', 'VariationID', 'PositionVCF', 'ReferenceAlleleVCF', 'AlternateAlleleVCF', 'SomaticClinicalImpact', 'SomaticClinicalImpactLastEvaluated', 'ReviewStatusClinicalImpact', 'Oncogenicity', 'OncogenicityLastEvaluated', 'ReviewStatusOncogenicity', 'SCVsForAggregateGermlineClassification', 'SCVsForAggregateSomaticClinicalImpact', 'SCVsForAggregateOncogenicityClassification']
                                                                                            0  \
#AlleleID                                                            

In [3]:
import pandas as pd

n0 = len(df)
print("start:", n0)

# 1. Primary fast boolean masks (Assembly, Origin, Type)
mask_base = (
    (df["Assembly"] == "GRCh38") &
    (df["OriginSimple"] == "germline") &
    (df["Type"] == "single nucleotide variant") &
    (~df["GeneSymbol"].fillna("").str.contains(";", regex=False))
)
d = df[mask_base].copy()
print("GRCh38 + germline + SNV + single-gene:", len(d))

# 2. Review Status (2+ stars, no conflicts)
rs = d["ReviewStatus"].fillna("")
mask_review = (
    rs.str.contains(r"multiple submitters|expert panel|practice guideline", case=False, regex=True) &
    ~rs.str.contains("conflicting", case=False, regex=False)
)
d = d[mask_review]
print("2+ star:", len(d))

# 3. Clinical Significance (Vectorized single-pass check)
cs = d["ClinicalSignificance"].fillna("").str.lower()
has_path = cs.str.contains("pathogenic", regex=False)
has_ben  = cs.str.contains("benign", regex=False)

# Mutually exclusive: either strictly Pathogenic (1) or strictly Benign (0)
valid_label = has_path ^ has_ben  # XOR: exactly one is True
d = d[valid_label].copy()
d["label"] = has_path[d.index].astype(int)
print("labelled:", len(d))
print(d["label"].value_counts())

# 4. Extract Missense Info (only on retained high-confidence rows)
extracted = d["Name"].str.extract(r"\(p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})\)")
d["wt3"] = extracted[0]
d["pos"] = extracted[1]
d["mut3"] = extracted[2]

# Missense: must have parsed coords, not nonsense, not synonymous
mask_missense = (
    d["wt3"].notna() &
    (d["mut3"] != "Ter") &
    (d["wt3"] != d["mut3"])
)
d = d[mask_missense]
print("missense:", len(d))

# 5. Deduplicate
d = d.drop_duplicates("VariationID")
print("final:", len(d))
print(d["label"].value_counts())

d.to_csv("filtered_missense_grch38.tsv", sep="\t", index=False)

start: 9311986
GRCh38 + germline + SNV + single-gene: 3984698
2+ star: 723635
labelled: 404410
label
0    338683
1     65727
Name: count, dtype: int64
missense: 66437
final: 66420
label
0    41162
1    25258
Name: count, dtype: int64


### Save the cleaned dataset

In [15]:
import hashlib
import json
from datetime import date
import pandas as pd

RAW = "variant_summary.txt.gz"
OUT = "clinvar_missense_grch38_clean.csv"
MANIFEST = "clinvar_missense_grch38_clean.manifest.json"

THREE_TO_ONE = {
    "Ala":"A","Arg":"R","Asn":"N","Asp":"D","Cys":"C","Gln":"Q","Glu":"E",
    "Gly":"G","His":"H","Ile":"I","Leu":"L","Lys":"K","Met":"M","Phe":"F",
    "Pro":"P","Ser":"S","Thr":"T","Trp":"W","Tyr":"Y","Val":"V",
}

counts = {}
df = pd.read_csv(RAW, sep="\t", low_memory=False).rename(columns={"#AlleleID": "AlleleID"})
counts["raw"] = len(df)

d = df[(df["Assembly"] == "GRCh38") &
       (df["OriginSimple"] == "germline") &
       (df["Type"] == "single nucleotide variant")]
counts["grch38_germline_snv"] = len(d)

rs = d["ReviewStatus"].fillna("")
d = d[rs.str.contains(r"multiple submitters|expert panel|practice guideline", case=False)
      & ~rs.str.contains("conflicting", case=False)]
counts["review_2plus_star"] = len(d)

cs = d["ClinicalSignificance"].fillna("").str.lower()
is_path = cs.str.contains("pathogenic") & ~cs.str.contains("benign")
is_ben = cs.str.contains("benign") & ~cs.str.contains("pathogenic")
d = d[is_path | is_ben].copy()
d["label"] = is_path[d.index].astype(int)
counts["labelled"] = len(d)

m = d["Name"].str.extract(r"\(p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})\)")
d = d.assign(wt3=m[0], pos=m[1], mut3=m[2]).dropna(subset=["wt3"])
d = d[(d["mut3"] != "Ter") & (d["wt3"] != d["mut3"])]
counts["missense"] = len(d)

d = d[~d["GeneSymbol"].astype(str).str.contains(";")]
d = d.drop_duplicates("VariationID")
d["wt"] = d["wt3"].map(THREE_TO_ONE)
d["mut"] = d["mut3"].map(THREE_TO_ONE)
d = d.dropna(subset=["wt", "mut"])
d["pos"] = d["pos"].astype(int)
counts["final"] = len(d)

keep = ["VariationID", "AlleleID", "GeneSymbol", "Name", "pos", "wt", "mut",
        "label", "ClinicalSignificance", "ReviewStatus", "NumberSubmitters",
        "Chromosome", "Start", "ReferenceAllele", "AlternateAllele", "RCVaccession"]
out = d[keep].sort_values("VariationID").reset_index(drop=True)
out.to_csv(OUT, index=False)

sha = hashlib.sha256(open(OUT, "rb").read()).hexdigest()
manifest = {
    "output": OUT,
    "sha256": sha,
    "created": date.today().isoformat(),
    "source": RAW,
    "filters": {
        "Assembly": "GRCh38",
        "OriginSimple": "germline",
        "Type": "single nucleotide variant",
        "ReviewStatus": "multiple submitters, expert panel, or practice guideline; no conflicts",
        "labels": "pathogenic/likely pathogenic vs benign/likely benign; conflicting or mixed excluded",
        "consequence": "missense parsed from Name; nonsense and synonymous removed",
        "genes": "single gene only",
    },
    "row_counts": counts,
    "label_counts": out["label"].value_counts().to_dict(),
    "n_genes": int(out["GeneSymbol"].nunique()),
}
with open(MANIFEST, "w") as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest, indent=2))

{
  "output": "clinvar_missense_grch38_clean.csv",
  "sha256": "95c706a213f3cb1098a0f37f99c14250bddbb2a9e5d562f51aebe6cced0c4d84",
  "created": "2026-10-23",
  "source": "variant_summary.txt.gz",
  "filters": {
    "Assembly": "GRCh38",
    "OriginSimple": "germline",
    "Type": "single nucleotide variant",
    "ReviewStatus": "multiple submitters, expert panel, or practice guideline; no conflicts",
    "labels": "pathogenic/likely pathogenic vs benign/likely benign; conflicting or mixed excluded",
    "consequence": "missense parsed from Name; nonsense and synonymous removed",
    "genes": "single gene only"
  },
  "row_counts": {
    "raw": 9311986,
    "grch38_germline_snv": 3995266,
    "review_2plus_star": 724580,
    "labelled": 405149,
    "missense": 66662,
    "final": 66345
  },
  "label_counts": {
    "0": 41141,
    "1": 25204
  },
  "n_genes": 7344
}


In [14]:
df.columns

Index(['AlleleID', 'Type', 'Name', 'GeneID', 'GeneSymbol', 'HGNC_ID',
       'ClinicalSignificance', 'ClinSigSimple', 'LastEvaluated', 'RS# (dbSNP)',
       'nsv/esv (dbVar)', 'RCVaccession', 'PhenotypeIDS', 'PhenotypeList',
       'Origin', 'OriginSimple', 'Assembly', 'ChromosomeAccession',
       'Chromosome', 'Start', 'Stop', 'ReferenceAllele', 'AlternateAllele',
       'Cytogenetic', 'ReviewStatus', 'NumberSubmitters', 'Guidelines',
       'TestedInGTR', 'OtherIDs', 'SubmitterCategories', 'VariationID',
       'PositionVCF', 'ReferenceAlleleVCF', 'AlternateAlleleVCF',
       'SomaticClinicalImpact', 'SomaticClinicalImpactLastEvaluated',
       'ReviewStatusClinicalImpact', 'Oncogenicity',
       'OncogenicityLastEvaluated', 'ReviewStatusOncogenicity',
       'SCVsForAggregateGermlineClassification',
       'SCVsForAggregateSomaticClinicalImpact',
       'SCVsForAggregateOncogenicityClassification'],
      dtype='object')

In [20]:
import gzip, re

seqs = {}
with gzip.open("human_canonical.fasta.gz", "rt") as f:
    cur_gene = None
    cur_seq = []
    for line in f:
        if line.startswith(">"):
            if cur_gene and cur_seq:
                seqs[cur_gene] = "".join(cur_seq)
            # Extract GN=GeneSymbol from FASTA header
            m = re.search(r"GN=([^\s]+)", line)
            cur_gene = m.group(1) if m else None
            cur_seq = []
        else:
            cur_seq.append(line.strip())
    if cur_gene and cur_seq:
        seqs[cur_gene] = "".join(cur_seq)


d = pd.read_csv("clinvar_missense_grch38_clean.csv")
d["seq"] = d["GeneSymbol"].map(seqs)
d = d.dropna(subset=["seq"])
d = d[d.seq.str.len() <= 1022].copy()

# 2. Safe, fast reference validation
seq_arr = d["seq"].to_numpy()
pos_arr = d["pos"].astype(int).to_numpy() - 1
wt_arr = d["wt"].to_numpy()

# Safe verification handling out-of-bounds pos
valid_pos = (pos_arr >= 0) & (pos_arr < [len(s) for s in seq_arr])
ref_match = [
    seq_arr[i][pos_arr[i]] == wt_arr[i] if valid_pos[i] else False
    for i in range(len(d))
]

d = d[ref_match].copy()
print("after reference check:", len(d))


after reference check: 33996


In [24]:
d.columns
d.head()

,VariationID,AlleleID,GeneSymbol,Name,pos,wt,mut,label,ClinicalSignificance,ReviewStatus,NumberSubmitters,Chromosome,Start,ReferenceAllele,AlternateAllele,RCVaccession,seq
0,9,15048,HFE,NM_000410.4(HFE):c.845G>A (p.Cys282Tyr),282,C,Y,1,"Pathogenic/Pathogenic, low penetrance; risk fa...","criteria provided, multiple submitters, no con...",56,6,26092913,na,na,RCV000000019|RCV000178096|RCV000210820|RCV0003...,MGPRARPALLLLMLLQTAVLQGRLLRSHSLHYLFMGASEQDLGLSL...
1,19,15058,HFE,NM_000410.4(HFE):c.848A>C (p.Gln283Pro),283,Q,P,1,Pathogenic/Likely pathogenic,"criteria provided, multiple submitters, no con...",5,6,26092916,na,na,RCV000000036|RCV001050090|RCV005229758|RCV0038...,MGPRARPALLLLMLLQTAVLQGRLLRSHSLHYLFMGASEQDLGLSL...
2,30,15069,HOGA1,NM_138413.4(HOGA1):c.860G>T (p.Gly287Val),287,G,V,1,Pathogenic,"criteria provided, multiple submitters, no con...",12,10,97611535,na,na,RCV000000047|RCV000798240,MLGPQVWSSVRQGLSRSLSRNVGVWASGEGKKVDIAGIYPPVTTPF...
3,31,15070,HOGA1,NM_138413.4(HOGA1):c.289C>T (p.Arg97Cys),97,R,C,1,Pathogenic/Likely pathogenic,"criteria provided, multiple submitters, no con...",12,10,97598852,na,na,RCV000000048|RCV001059730|RCV003415591|RCV0068...,MLGPQVWSSVRQGLSRSLSRNVGVWASGEGKKVDIAGIYPPVTTPF...
4,33,15072,HOGA1,NM_138413.4(HOGA1):c.209G>C (p.Arg70Pro),70,R,P,1,Pathogenic/Likely pathogenic,"criteria provided, multiple submitters, no con...",4,10,97584912,na,na,RCV000000050|RCV001387338,MLGPQVWSSVRQGLSRSLSRNVGVWASGEGKKVDIAGIYPPVTTPF...


In [25]:
three_to_one = {
    "Ala":"A","Arg":"R","Asn":"N","Asp":"D","Cys":"C","Gln":"Q","Glu":"E",
    "Gly":"G","His":"H","Ile":"I","Leu":"L","Lys":"K","Met":"M","Phe":"F",
    "Pro":"P","Ser":"S","Thr":"T","Trp":"W","Tyr":"Y","Val":"V"
}
d = d.copy()
# d["wt"] = d["wt"].map(three_to_one)
# d["mut"] = d["mut"].map(three_to_one)
d["pos"] = d["pos"].astype(int)
d = d.dropna(subset=["wt", "mut"])
print("Final no. of rows: ", len(d))
d[["VariationID", "GeneSymbol", "pos", "wt", "mut", "label"]].to_csv(
    "parsed_missense.tsv", sep="\t", index=False)

Final no. of rows:  33996


### Score with ESM-2 (zero-shot and embeddings)

In [ ]:
import numpy as np, torch
from transformers import AutoTokenizer, EsmForMaskedLM

name = "facebook/esm2_t33_650M_UR50D"
tok = AutoTokenizer.from_pretrained(name)
model = EsmForMaskedLM.from_pretrained(name).half().cuda().eval()

AA = list("ACDEFGHIKLMNPQRSTVWY")
aa_id = {a: tok.convert_tokens_to_ids(a) for a in AA}
MASK = tok.mask_token_id

def enc(seq):
    # Token 0 is <cls>, so residue i (1-based) sits at index i
    return tok(seq, return_tensors="pt").input_ids.cuda()

n = len(d)
zs = np.full(n, np.nan, dtype=np.float32)
emb = np.zeros((n, 1280), dtype=np.float16)
idx_of = {ix: k for k, ix in enumerate(d.index)}

with torch.inference_mode():
    for gene, g in d.groupby("GeneSymbol"):
        seq = g.seq.iloc[0]
        h_wt = model(enc(seq), output_hidden_states=True).hidden_states[-1][0]

        # Embedding difference: mutant minus wild-type at the mutated position
        for ix, row in g.iterrows():
            p, m = row.pos, row.mut
            mut_seq = seq[:p - 1] + m + seq[p:]
            h = model(enc(mut_seq), output_hidden_states=True).hidden_states[-1][0]
            emb[idx_of[ix]] = (h[p] - h_wt[p]).float().cpu().numpy()

        # Zero-shot masked-marginal score: one masked pass per unique position
        for p, gp in g.groupby("pos"):
            ids = enc(seq)
            ids[0, p] = MASK
            logp = torch.log_softmax(model(ids).logits[0, p].float(), dim=-1)
            for ix, row in gp.iterrows():
                zs[idx_of[ix]] = (logp[aa_id[row.mut]] - logp[aa_id[row.wt]]).item()

assert not np.isnan(zs).any()
np.save("esm_emb.npy", emb)
np.save("esm_zs.npy", zs)
d.drop(columns="seq").to_csv("scored_missense.tsv", sep="\t", index=False)
print("saved", n, "variants")

In [ ]:
name = "facebook/esm2_t33_650M_UR50D"
tok = AutoTokenizer.from_pretrained(name)
model = EsmForMaskedLM.from_pretrained(name).half().cuda().eval()
esm = model.esm
MASK = tok.mask_token_id

AA = list("ACDEFGHIKLMNPQRSTVWY")
aa_id = torch.tensor([tok.convert_tokens_to_ids(a) for a in AA], device="cuda")
aa_idx = {a: i for i, a in enumerate(AA)}

seqs = json.load(open("uniprot_seqs.json"))
d = pd.read_csv("scored_missense.tsv", sep="\t")
d["seq"] = d.GeneSymbol.map(seqs)
n = len(d)
emb = np.zeros((n, 1280), dtype=np.float16)
zs = np.full(n, np.nan, dtype=np.float32)
BS = 32

@torch.inference_mode()
def hidden(strs):
    # Right padding, so residue p (1-based) is at index p in every row
    enc = tok(strs, return_tensors="pt", padding=True).to("cuda")
    return esm(**enc).last_hidden_state          # (B, L, 1280)

@torch.inference_mode()
def masked_logp(seq, positions):
    base = tok(seq, return_tensors="pt").input_ids.cuda()[0]
    P = torch.tensor(positions, device="cuda")
    batch = base.repeat(len(P), 1)
    batch[torch.arange(len(P), device="cuda"), P] = MASK
    logits = model(input_ids=batch).logits[torch.arange(len(P), device="cuda"), P]
    return torch.log_softmax(logits.float(), dim=-1)[:, aa_id].cpu().numpy()  # (U, 20)

for k, (gene, g) in enumerate(d.groupby("GeneSymbol")):
    seq = g.seq.iloc[0]
    ix = g.index.values
    p = g.pos.values                              # 1-based
    wt_i = np.array([aa_idx[a] for a in g.wt.values])
    mt_i = np.array([aa_idx[a] for a in g.mut.values])

    # Embeddings: wild-type once per gene, mutants in batches
    h_wt = hidden([seq])[0]                       # (L, 1280)
    for s in range(0, len(g), BS):
        sl = slice(s, s + BS)
        muts = [seq[:pp - 1] + m + seq[pp:] for pp, m in zip(p[sl], g.mut.values[sl])]
        h = hidden(muts)
        pt = torch.tensor(p[sl], device="cuda")
        rows = torch.arange(len(muts), device="cuda")
        diff = h[rows, pt] - h_wt[pt]
        emb[ix[sl]] = diff.float().cpu().numpy()

    # Zero-shot: one masked pass per unique position, batched
    uniq = np.unique(p)
    M = np.concatenate([masked_logp(seq, uniq[s:s + BS].tolist())
                        for s in range(0, len(uniq), BS)], axis=0)  # (U, 20)
    u = np.searchsorted(uniq, p)
    zs[ix] = M[u, mt_i] - M[u, wt_i]

    if k % 200 == 0:
        print(f"genes done: {k}", flush=True)

assert not np.isnan(zs).any()
np.save("esm_emb.npy", emb)
np.save("esm_zs.npy", zs)
print("saved", n, "variants")

#### Modified optimized code

In [30]:
df = pd.read_csv("cleaned_missense.csv")
d = df.head(1000).reset_index(drop=True)
print(len(d))

1000


#### Load all protein sequences

In [34]:
seqs = {}
with gzip.open("human_canonical.fasta.gz", "rt") as f:
    cur_gene = None
    cur_seq = []
    for line in f:
        if line.startswith(">"):
            if cur_gene and cur_seq:
                seqs[cur_gene] = "".join(cur_seq)
            m = re.search(r"GN=([^\s]+)", line)
            cur_gene = m.group(1) if m else None
            cur_seq = []
        else:
            cur_seq.append(line.strip())
    if cur_gene and cur_seq:
        seqs[cur_gene] = "".join(cur_seq)

print(f"    Loaded {len(seqs)} canonical sequences from FASTA.")

    Loaded 20394 canonical sequences from FASTA.


In [ ]:
name = "facebook/esm2_t33_650M_UR50D"
tok = AutoTokenizer.from_pretrained(name)
model = EsmForMaskedLM.from_pretrained(name).half().cuda().eval()
esm = model.esm
MASK = tok.mask_token_id

AA = list("ACDEFGHIKLMNPQRSTVWY")
aa_id = torch.tensor([tok.convert_tokens_to_ids(a) for a in AA], device="cuda")
aa_idx = {a: i for i, a in enumerate(AA)}

df = pd.read_csv("cleaned_missense.csv")
# d = df.head(1000).reset_index(drop=True)
d = df.copy()
d["seq"] = d.GeneSymbol.map(seqs)

n = len(d)
emb = np.zeros((n, 1280), dtype=np.float16)
zs = np.full(n, np.nan, dtype=np.float32)
BS = 32

@torch.inference_mode()
def hidden(strs):
    enc = tok(strs, return_tensors="pt", padding=True).to("cuda")
    return esm(**enc).last_hidden_state

@torch.inference_mode()
def masked_logp(seq, positions):
    base = tok(seq, return_tensors="pt").input_ids.cuda()[0]
    P = torch.tensor(positions, device="cuda")
    batch = base.repeat(len(P), 1)
    batch[torch.arange(len(P), device="cuda"), P] = MASK
    logits = model(input_ids=batch).logits[torch.arange(len(P), device="cuda"), P]
    return torch.log_softmax(logits.float(), dim=-1)[:, aa_id].cpu().numpy()

# Progress tracking over genes
genes_grouped = list(d.groupby("GeneSymbol"))
pbar = tqdm(genes_grouped, desc="Processing Genes", unit="gene")

for gene, g in pbar:
    seq = g.seq.iloc[0]
    ix = g.index.values
    p = g.pos.values
    wt_i = np.array([aa_idx[a] for a in g.wt.values])
    mt_i = np.array([aa_idx[a] for a in g.mut.values])

    # 1. WT Hidden Representation
    h_wt = hidden([seq])[0]

    # 2. Mutant Embeddings
    for s in range(0, len(g), BS):
        sl = slice(s, s + BS)
        muts = [seq[:pp - 1] + m + seq[pp:] for pp, m in zip(p[sl], g.mut.values[sl])]
        h = hidden(muts)
        pt = torch.tensor(p[sl], device="cuda")
        rows = torch.arange(len(muts), device="cuda")
        diff = h[rows, pt] - h_wt[pt]
        emb[ix[sl]] = diff.float().cpu().numpy()

    # 3. Masked Zero-Shot Scoring
    uniq = np.unique(p)
    M = np.concatenate([
        masked_logp(seq, uniq[s:s + BS].tolist())
        for s in range(0, len(uniq), BS)
    ], axis=0)
    u = np.searchsorted(uniq, p)
    zs[ix] = M[u, mt_i] - M[u, wt_i]

assert not np.isnan(zs).any()
np.save("esm_emb.npy", emb)
np.save("esm_zs.npy", zs)
print(f"Successfully saved {n} variants.")

### Gene-grouped split and evaluation

In [42]:
df = pd.read_csv("cleaned_missense.csv")
d = df.copy()
# d = df.head(1000).reset_index(drop=True)
emb = np.load("esm_emb.npy").astype(np.float32)
zs = np.load("esm_zs.npy")
y = d.label.values
groups = d.GeneSymbol.values

# Gene-disjoint split, stratified by label
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
tr, te = next(sgkf.split(emb, y, groups))
assert set(groups[tr]).isdisjoint(groups[te])
print(f"train {len(tr)} (pos rate {y[tr].mean():.3f}), "
      f"test {len(te)} (pos rate {y[te].mean():.3f})")

# Baseline: BLOSUM62 (lower score = more dissimilar = more likely damaging)
B = substitution_matrices.load("BLOSUM62")
blosum = np.array([B[w][m] for w, m in zip(d.wt, d.mut)])

# Classifier trained on the training genes only
scaler = StandardScaler().fit(emb[tr])
X = scaler.transform(emb)
clf = LogisticRegression(C=0.1, max_iter=2000, class_weight="balanced")
clf.fit(X[tr], y[tr])
p_clf = clf.decision_function(X)

def report(name, score):
    print(f"{name:30s} AUROC={roc_auc_score(y[te], score[te]):.3f}  "
          f"PR-AUC={average_precision_score(y[te], score[te]):.3f}")

print(f"PR-AUC of random guessing = {y[te].mean():.3f}")
report("BLOSUM62 (-score)", -blosum)
report("ESM-2 zero-shot (-LLR)", -zs)
report("ESM-2 embedding + logreg", p_clf)

train 27545 (pos rate 0.492), test 6451 (pos rate 0.424)
PR-AUC of random guessing = 0.424
BLOSUM62 (-score)              AUROC=0.644  PR-AUC=0.537
ESM-2 zero-shot (-LLR)         AUROC=0.885  PR-AUC=0.866
ESM-2 embedding + logreg       AUROC=0.868  PR-AUC=0.834


In [45]:
filt = pd.read_csv("filtered_missense_grch38.tsv", sep="\t", low_memory=False)
parsed = pd.read_csv("parsed_missense.tsv", sep="\t")
scored = pd.read_csv("cleaned_missense.csv")

print("filtered:", len(filt), "| unique IDs:", filt.VariationID.nunique())
print("parsed:  ", len(parsed), "| unique IDs:", parsed.VariationID.nunique())
print("scored:  ", len(scored), "| unique IDs:", scored.VariationID.nunique())

dropped = filt[~filt.VariationID.isin(scored.VariationID)]
print("\ndropped between filtering and scoring:", len(dropped))
print("positive rate, filtered:", round(filt.label.mean(), 3))
print("positive rate, dropped: ", round(dropped.label.mean(), 3) if len(dropped) else "n/a")
print("positive rate, scored:  ", round(scored.label.mean(), 3))
print(dropped.GeneSymbol.value_counts().head(15))

filtered: 66420 | unique IDs: 66420
parsed:   33996 | unique IDs: 33996
scored:   33996 | unique IDs: 33996

dropped between filtering and scoring: 32424
positive rate, filtered: 0.38
positive rate, dropped:  0.277
positive rate, scored:   0.479
GeneSymbol
FBN1      629
TTN       389
BRCA1     336
ABCA4     335
BRCA2     309
SCN1A     307
KMT2D     292
NF1       286
COL4A5    275
USH2A     230
COL1A2    213
MYH7      202
CFTR      198
MECP2     188
RYR1      188
Name: count, dtype: int64


In [4]:
import json
import os
import time
import pandas as pd
import requests
from tqdm import tqdm

CACHE = "uniprot_seqs.json"
BATCH_SIZE = 50  # Fetch 50 genes per HTTP roundtrip

# 1. Load only the necessary column
df = pd.read_csv("parsed_missense.tsv", sep="\t", usecols=["GeneSymbol"])
all_genes = sorted(df["GeneSymbol"].dropna().unique())

# 2. Load existing cache or initialize
seqs = {}
if os.path.exists(CACHE):
    with open(CACHE, "r") as f:
        seqs = json.load(f)
    print(f"Loaded cache with {len(seqs)} genes")

# Determine remaining genes to query
genes_to_fetch = [g for g in all_genes if g not in seqs]
print(f"Total genes: {len(all_genes)} | Already cached: {len(seqs)} | To fetch: {len(genes_to_fetch)}")

def fetch_batch(session, gene_batch, max_retries=3):
    gene_query = " OR ".join(f"gene_exact:{g}" for g in gene_batch)
    query = f"({gene_query}) AND organism_id:9606 AND reviewed:true"
    
    params = {
        "query": query,
        "fields": "gene_names,sequence",
        "format": "json",
        "size": 500  # Plenty of buffer for isoforms/synonyms matching the batch
    }
    
    err = None
    for attempt in range(max_retries):
        try:
            r = session.get(
                "https://rest.uniprot.org/uniprotkb/search",
                params=params,
                timeout=30
            )
            r.raise_for_status()
            return r.json().get("results", []), None
        except requests.RequestException as e:
            err = str(e)
            time.sleep(2 ** attempt)  # Exponential backoff (1s, 2s, 4s)
            
    return [], err

# 3. Fetch in batches with live progress tracking
errors = {}
if genes_to_fetch:
    with requests.Session() as session:
        with tqdm(total=len(genes_to_fetch), desc="Fetching UniProt Seqs", unit="genes") as pbar:
            for i in range(0, len(genes_to_fetch), BATCH_SIZE):
                batch = genes_to_fetch[i : i + BATCH_SIZE]
                results, err = fetch_batch(session, batch)
                
                if err:
                    for g in batch:
                        errors[g] = err
                else:
                    # Map retrieved entries back to the requested gene symbols
                    batch_found = {}
                    for entry in results:
                        seq = entry.get("sequence", {}).get("value")
                        for g_data in entry.get("genes", []):
                            primary_gene = g_data.get("geneName", {}).get("value")
                            if primary_gene and primary_gene in batch and primary_gene not in batch_found:
                                batch_found[primary_gene] = seq

                    for g in batch:
                        seqs[g] = batch_found.get(g, None)

                # Checkpoint cache every 500 genes to protect against crashes
                if (i // BATCH_SIZE) % 10 == 0:
                    with open(CACHE, "w") as f:
                        json.dump(seqs, f)

                # Advance progress bar by batch count
                pbar.update(len(batch))

    # Final write to cache
    with open(CACHE, "w") as f:
        json.dump(seqs, f)

# Summary
missing = [g for g in all_genes if not seqs.get(g)]
print(f"\nDone.")
print(f"Total genes with sequence found: {len(all_genes) - len(missing)}")
print(f"Genes with no sequence: {len(missing)}")
print(f"Genes with network/HTTP errors: {len(errors)}")
if errors:
    print("Sample errors:", list(errors.items())[:3])

Loaded cache with 4050 genes
Total genes: 7344 | Already cached: 4050 | To fetch: 3294


Fetching UniProt Seqs: 100%|██████████| 3294/3294 [00:37<00:00, 88.72genes/s] 


Done.
Total genes with sequence found: 7279
Genes with no sequence: 65
Genes with network/HTTP errors: 0


In [3]:
three = {
    "Ala":"A","Arg":"R","Asn":"N","Asp":"D","Cys":"C","Gln":"Q","Glu":"E",
    "Gly":"G","His":"H","Ile":"I","Leu":"L","Lys":"K","Met":"M","Phe":"F",
    "Pro":"P","Ser":"S","Thr":"T","Trp":"W","Tyr":"Y","Val":"V"
}

filt = pd.read_csv("filtered_missense_grch38.tsv", sep="\t", low_memory=False)
print("filtered rows:", len(filt), "| unique IDs:", filt.VariationID.nunique())

m = filt.Name.str.extract(r"\(p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})\)")
parsed = filt.assign(wt=m[0].map(three), pos=pd.to_numeric(m[1]), mut=m[2].map(three))
n0 = len(parsed)
parsed = parsed.dropna(subset=["wt", "mut", "pos"])
print("dropped unparseable:", n0 - len(parsed))

parsed = parsed[["VariationID", "GeneSymbol", "pos", "wt", "mut", "label"]].copy()
parsed["pos"] = parsed["pos"].astype(int)
parsed.to_csv("parsed_missense.tsv", sep="\t", index=False)

print("parsed rows:", len(parsed), "| unique IDs:", parsed.VariationID.nunique())
print("positive rate:", round(parsed.label.mean(), 3))

filtered rows: 66420 | unique IDs: 66420
dropped unparseable: 75
parsed rows: 66345 | unique IDs: 66345
positive rate: 0.38


# Date: Oct 23

### Check whether parsed_missense.tsv is okay before we run analysis

In [10]:
p = pd.read_csv("parsed_missense.tsv", sep="\t")
print("shape:", p.shape)
print("columns:", p.columns.tolist())
print("\nmissing values per column:\n", p.isna().sum())
print("\nunique IDs:", p.VariationID.nunique(), "of", len(p))
print("duplicate IDs:", p.VariationID.duplicated().sum())
print("\nlabel counts:\n", p.label.value_counts())
print("positive rate:", round(p.label.mean(), 3))
print("\nwt values:", sorted(p.wt.dropna().unique()))
print("mut values:", sorted(p.mut.dropna().unique()))
print("wt == mut rows:", (p.wt == p.mut).sum())
print("pos range:", p.pos.min(), "-", p.pos.max())

seqs = json.load(open("uniprot_seqs.json"))
genes = p.GeneSymbol.dropna().unique()
print("\ngenes in parsed:", len(genes))
print("genes missing from cache:", sum(1 for g in genes if not seqs.get(g)))

shape: (66345, 6)
columns: ['VariationID', 'GeneSymbol', 'pos', 'wt', 'mut', 'label']

missing values per column:
 VariationID    0
GeneSymbol     0
pos            0
wt             0
mut            0
label          0
dtype: int64

unique IDs: 66345 of 66345
duplicate IDs: 0

label counts:
 label
0    41141
1    25204
Name: count, dtype: int64
positive rate: 0.38

wt values: ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']
mut values: ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']
wt == mut rows: 0
pos range: 1 - 35956

genes in parsed: 7344
genes missing from cache: 65


## Diagnose the dropped set

### Find the exact reasons why nearly half of the variants were dropped

In [ ]:
import json, pandas as pd

p = pd.read_csv("parsed_missense.tsv", sep="\t")
seqs = json.load(open("uniprot_seqs.json"))
p["seq"] = p.GeneSymbol.map(seqs)

def reason(r):
    if not isinstance(r.seq, str) or len(r.seq) == 0:
        return "no_sequence"
    if len(r.seq) > 1022:
        return "too_long"
    if r.pos < 1 or r.pos > len(r.seq):
        return "pos_out_of_range"
    if r.seq[r.pos - 1] != r.wt:
        return "ref_mismatch"
    return "ok"

p["reason"] = p.apply(reason, axis=1)

print(p.reason.value_counts())
print("\npositive rate by reason:")
print(p.groupby("reason").label.agg(["count", "mean"]).round(3))

for r in ["pos_out_of_range", "ref_mismatch", "too_long"]:
    print(f"\n{r} genes:")
    print(p.loc[p.reason == r, "GeneSymbol"].value_counts().head(10))

p.drop(columns="seq").to_csv("reasons_missense.tsv", sep="\t", index=False)
print("\nsaved reasons_missense.tsv")

parse_reason
ok          66345
unparsed       75
Name: count, dtype: int64
parsed rows: 66345


### Parser to find the reason behind missing rows 

In [16]:
f = pd.read_csv("filtered_missense_grch38.tsv", sep="\t", low_memory=False)
m = f.Name.str.extract(r"\(p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})\)")
f = f.assign(wt=m[0].map(three), pos=pd.to_numeric(m[1]), mut=m[2].map(three))

f["parse_reason"] = "ok"
f.loc[f.Name.str.contains(r"\(p\.") == False, "parse_reason"] = "no_protein_change"
f.loc[f.wt.isna() & f.parse_reason.eq("ok"), "parse_reason"] = "unparsed"
f.loc[f.wt.eq("*"), "parse_reason"] = "stop_loss"
f.loc[f.mut.eq("*"), "parse_reason"] = "nonsense"
f.loc[f.wt.eq(f.mut) & f.parse_reason.eq("ok"), "parse_reason"] = "synonymous"

print(f.parse_reason.value_counts())
f[f.parse_reason != "ok"][["VariationID","GeneSymbol","Name","parse_reason"]].to_csv(
    "parse_excluded.tsv", sep="\t", index=False)

kept = f[f.parse_reason == "ok"][["VariationID","GeneSymbol","pos","wt","mut","label"]].copy()
kept["pos"] = kept.pos.astype(int)
kept.to_csv("parsed_missense.tsv", sep="\t", index=False)
print("parsed rows:", len(kept))

parse_reason
ok          66345
unparsed       75
Name: count, dtype: int64
parsed rows: 66345


### Why counts don't add up? Some rows (75) are not being parsed properly

In [14]:
f = pd.read_csv("filtered_missense_grch38.tsv", sep="\t", low_memory=False)
p = pd.read_csv("parsed_missense.tsv", sep="\t")
r = pd.read_csv("reasons_missense.tsv", sep="\t")
sc = pd.read_csv("cleaned_missense.csv")

print("Filtered:", len(f), "| Parsed:", len(p), "| Reasons:", len(r), "| Scored:", len(sc))
missing = f[~f.VariationID.isin(p.VariationID)]
print("in filtered, not parsed:", len(missing))
print(missing.Name.head(10).tolist())

ok_ids = set(r.loc[r.reason == "ok", "VariationID"])
print("Ok:", len(ok_ids), "| Scored but not ok:", (~sc.VariationID.isin(ok_ids)).sum())

Filtered: 66420 | Parsed: 66345 | Reasons: 66345 | Scored: 33996
in filtered, not parsed: 75
['NM_001368894.2(PAX6):c.1310A>T (p.Ter437Leu)', 'NM_024301.5(FKRP):c.1486T>A (p.Ter496Arg)', 'NM_032551.5(KISS1R):c.1195T>A (p.Ter399Arg)', 'NM_004531.5(MOCS2):c.567A>C (p.Ter189Tyr)', 'NM_000203.4(IDUA):c.1960T>G (p.Ter654Gly)', 'NM_000517.4(HBA2):c.427T>A (p.Ter143Lys)', 'NM_000517.6(HBA2):c.428A>C (p.Ter143Ser)', 'NM_000517.4(HBA2):c.429A>T (p.Ter143Tyr)', 'NM_000517.4(HBA2):c.427T>G (p.Ter143Glu)', 'NM_000142.5(FGFR3):c.2419T>G (p.Ter807Gly)']
Ok: 33955 | Scored but not ok: 86


### Reason behind dropping the unparsed rows

In [20]:
sc = pd.read_csv("scored_missense.tsv", sep="\t")
r = pd.read_csv("reasons_missense.tsv", sep="\t")
seqs = json.load(open("uniprot_seqs.json"))

r = r[["VariationID", "reason"]]
x = sc[["VariationID", "GeneSymbol", "pos", "wt", "mut"]].merge(r, on="VariationID", how="left")
bad = x[x.reason != "ok"].copy()

print("scored but not ok, by reason:")
print(bad.reason.value_counts(dropna=False))

bad["seq_len"] = bad.GeneSymbol.map(lambda g: len(seqs.get(g) or ""))
bad["aa_at_pos"] = [
    seqs[g][p-1] if seqs.get(g) and 0 < p <= len(seqs[g]) else None
    for g, p in zip(bad.GeneSymbol, bad.pos)
]
print(bad.head(20).to_string(index=False))
print("\ngenes:")
print(bad.GeneSymbol.value_counts().head(10))

not_scored = r[r.reason == "ok"].merge(sc[["VariationID"]], on="VariationID", how="left", indicator=True)
print("\nok but not scored:", (not_scored._merge == "left_only").sum())

scored but not ok, by reason:
reason
no_sequence         81
pos_out_of_range     5
Name: count, dtype: int64
 VariationID GeneSymbol  pos wt mut      reason  seq_len aa_at_pos
       11115        WAS   86  R   H no_sequence        0      None
       11116        WAS   56  A   V no_sequence        0      None
       11123        WAS   45  T   M no_sequence        0      None
       11125        WAS  270  L   P no_sequence        0      None
       29967        WAS  294  I   T no_sequence        0      None
      130855      ZNF81  117  G   V no_sequence        0      None
      130857      ZNF81  185  S   L no_sequence        0      None
      130858      ZNF81  213  A   E no_sequence        0      None
      130859      ZNF81    3  A   V no_sequence        0      None
      135410        WAS  332  V   A no_sequence        0      None
      135411        WAS  460  P   S no_sequence        0      None
      212693     ZNF711  277  A   T no_sequence        0      None
      212695      ZN

## Fix the cache

In [22]:
import json, time, requests, pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

CACHE = "uniprot_seqs.json"
seqs = json.load(open(CACHE))
p = pd.read_csv("parsed_missense.tsv", sep="\t")

need = sorted(g for g in p.GeneSymbol.dropna().unique() if not seqs.get(g))
print("genes needing a sequence:", len(need))

def query(gene, reviewed):
    q = f'gene_exact:"{gene}" AND organism_id:9606' + (" AND reviewed:true" if reviewed else "")
    r = requests.get("https://rest.uniprot.org/uniprotkb/search",
                     params={"query": q, "fields": "accession,sequence",
                             "format": "json", "size": 1}, timeout=60)
    if r.status_code == 429:
        time.sleep(15)
    r.raise_for_status()
    return r.json().get("results", [])

def fetch(gene):
    err = None
    for attempt in range(4):
        try:
            res = query(gene, reviewed=True) or query(gene, reviewed=False)
            if res:
                return gene, res[0]["sequence"]["value"], None
            return gene, None, "no UniProt entry"
        except Exception as e:
            err = repr(e)
            time.sleep(3 * (attempt + 1))
    return gene, None, err

failed = {}
ok = 0
with ThreadPoolExecutor(2) as ex:
    futures = [ex.submit(fetch, g) for g in need]
    bar = tqdm(as_completed(futures), total=len(futures), desc="UniProt", unit="gene")
    for fut in bar:
        gene, seq, err = fut.result()
        if seq:
            seqs[gene] = seq
            ok += 1
        else:
            failed[gene] = err
        bar.set_postfix(ok=ok, failed=len(failed))

json.dump(seqs, open(CACHE, "w"))
print(f"\nfetched: {ok} | still missing: {len(failed)}")
for g, e in list(failed.items())[:10]:
    print(" ", g, "->", e)

genes needing a sequence: 5


UniProt:   0%|          | 0/5 [00:00<?, ?gene/s]


fetched: 0 | still missing: 5
  IL33R -> no UniProt entry
  FAABTP -> no UniProt entry
  IL36R -> no UniProt entry
  covers 10 genes, none of which curated to show dosage sensitivity -> no UniProt entry
  covers 17 genes, none of which curated to show dosage sensitivity -> no UniProt entry


### Windowed scoring script for LONG sequences

In [ ]:
MAXRES = 1022          # ESM-2 limit (1024 tokens including <cls> and <eos>)
BS = 16                # lower to 8 if you hit out-of-memory errors
NAME = "facebook/esm2_t33_650M_UR50D"
MAX_MISSING = 10       # stop if more genes lack sequences than expected

# ---------- 1. Load data and sequences ----------
d = pd.read_csv("parsed_missense.tsv", sep="\t")
d["pos"] = pd.to_numeric(d["pos"])

seqs = json.load(open("uniprot_seqs.json"))
d["seq"] = d.GeneSymbol.map(seqs)

# Filter missing sequences
missing = d[d.seq.isna() | (d.seq.str.len() == 0)]
if len(missing):
    missing.to_csv("missing_seq.tsv", sep="\t", index=False)
    print(f"no sequence for {len(missing)} variants ({missing.GeneSymbol.nunique()} genes), logged to missing_seq.tsv")
assert missing.GeneSymbol.nunique() <= MAX_MISSING, "more missing genes than expected; check the cache"
d = d.drop(missing.index).reset_index(drop=True)

# Drop any coordinates that are strictly out of range for the retrieved sequence
L = d.seq.str.len().values
out_of_bounds = (d.pos <= 0) | (d.pos > L)
if out_of_bounds.any():
    bad_pos = d[out_of_bounds]
    bad_pos.drop(columns=["seq"]).to_csv("pos_out_of_bounds.tsv", sep="\t", index=False)
    print(f"out of bounds coordinates: {len(bad_pos)} variants, logged to pos_out_of_bounds.tsv")
    d = d[~out_of_bounds].reset_index(drop=True)
    L = d.seq.str.len().values  # recompute after drop

# ---------- 2. Choose windows ----------
def window(L, pos):
    """Return (start, end, local_pos) for a 1-based pos. local_pos is 1-based within the window."""
    if L <= MAXRES:
        return 0, L, pos
    start = min(max(pos - 1 - MAXRES // 2, 0), L - MAXRES)
    return start, start + MAXRES, pos - start

wins = [window(l, p) for l, p in zip(L, d.pos.values)]
d["w_start"] = [w[0] for w in wins]
d["w_end"] = [w[1] for w in wins]
d["lpos"] = [w[2] for w in wins]
d["full_len"] = L
d["windowed"] = (L > MAXRES).astype(int)
d["wseq"] = [s[a:b] for s, a, b in zip(d.seq, d.w_start, d.w_end)]

# Safe wild-type check
def check_wt(row):
    idx = int(row.lpos) - 1
    w = row.wseq
    return 0 <= idx < len(w) and w[idx] == row.wt


oob = d.lpos.values > d.wseq.str.len().values
if oob.any():
    d[oob].drop(columns=["seq", "wseq"]).to_csv("pos_out_of_range.tsv", sep="\t", index=False)
    print(f"position beyond sequence: {oob.sum()} variants logged to pos_out_of_range.tsv")
d = d[~oob].reset_index(drop=True)


ok = d.apply(check_wt, axis=1).values
bad = d[~ok]
if len(bad):
    bad.drop(columns=["seq", "wseq"]).to_csv("ref_fail_windowed.tsv", sep="\t", index=False)
    print(f"wild-type mismatch in window: {len(bad)} variants, logged to ref_fail_windowed.tsv")

d = d[ok].reset_index(drop=True)
print("variants to score:", len(d), "| windowed:", int(d.windowed.sum()))

tok = AutoTokenizer.from_pretrained(NAME)
model = EsmForMaskedLM.from_pretrained(NAME).half().cuda().eval()
esm = model.esm
MASK = tok.mask_token_id
AA = list("ACDEFGHIKLMNPQRSTVWY")
aa_id = torch.tensor([tok.convert_tokens_to_ids(a) for a in AA], device="cuda")
aa_idx = {a: i for i, a in enumerate(AA)}

@torch.inference_mode()
def hidden(strs):
    enc = tok(strs, return_tensors="pt", padding=True).to("cuda")
    return esm(**enc).last_hidden_state            # (B, T, 1280); residue p at index p

@torch.inference_mode()
def masked_logp(wseq, positions):
    base = tok(wseq, return_tensors="pt").input_ids.cuda()[0]
    P = torch.tensor(positions, device="cuda")
    batch = base.repeat(len(P), 1)
    rows = torch.arange(len(P), device="cuda")
    batch[rows, P] = MASK
    logits = model(input_ids=batch).logits[rows, P]
    return torch.log_softmax(logits.float(), dim=-1)[:, aa_id].cpu().numpy()   # (U, 20)

n = len(d)
emb = np.zeros((n, 1280), dtype=np.float16)
zs = np.full(n, np.nan, dtype=np.float32)

groups = d.groupby(["GeneSymbol", "w_start"], sort=False)
for (gene, ws), g in tqdm(groups, total=groups.ngroups, desc="scoring", unit="window"):
    wseq = g.wseq.iloc[0]
    ix = g.index.values
    lp = g.lpos.values
    mt = g.mut.values
    wt_i = np.array([aa_idx[a] for a in g.wt.values])
    mt_i = np.array([aa_idx[a] for a in mt])

    # Embeddings: wild-type window once, mutants in batches
    h_wt = hidden([wseq])[0]                                   # (T, 1280)
    for s in range(0, len(g), BS):
        sl = slice(s, s + BS)
        muts = [wseq[:p - 1] + m + wseq[p:] for p, m in zip(lp[sl], mt[sl])]
        h = hidden(muts)
        pt = torch.tensor(lp[sl], device="cuda")
        rows = torch.arange(len(muts), device="cuda")
        diff = h[rows, pt] - h_wt[pt]
        emb[ix[sl]] = diff.float().cpu().numpy()

    # Zero-shot: one masked pass per unique local position
    uniq = np.unique(lp)
    M = np.concatenate([masked_logp(wseq, uniq[s:s + BS].tolist())   for s in range(0, len(uniq), BS)], axis=0)
    u = np.searchsorted(uniq, lp)
    zs[ix] = M[u, mt_i] - M[u, wt_i]

assert not np.isnan(zs).any(), "some variants were not scored"

out = d.drop(columns=["seq", "wseq"])
out.to_csv("scored_windowed.tsv", sep="\t", index=False)
np.save("esm_emb_w.npy", emb)
np.save("esm_zs_w.npy", zs)
print("saved", n, "variants")

print(out.groupby("windowed").label.agg(n="count", pos_rate="mean").round(3))

no sequence for 13 variants (5 genes), logged to missing_seq.tsv
out of bounds coordinates: 233 variants, logged to pos_out_of_bounds.tsv
wild-type mismatch in window: 4198 variants, logged to ref_fail_windowed.tsv
variants to score: 61901 | windowed: 27858


Loading weights:   0%|          | 0/571 [00:00<?, ?it/s]

EsmForMaskedLM LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     |  | 
----------------------------+------------+--+-
esm.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


scoring:   0%|          | 0/19896 [00:00<?, ?window/s]

saved 61901 variants
              n  pos_rate
windowed                 
0         34043     0.479
1         27858     0.285


### 61901 out of 66345 variants ran, what happened to the others??

In [25]:
p = pd.read_csv("parsed_missense.tsv", sep="\t")
s = pd.read_csv("scored_windowed.tsv", sep="\t")

dropped = p[~p.VariationID.isin(s.VariationID)]
print("dropped:", len(dropped), "| positive rate:", round(dropped.label.mean(), 3))
print(dropped.GeneSymbol.value_counts().head(10))

for f in ["missing_seq.tsv", "ref_fail_windowed.tsv"]:
    try:
        x = pd.read_csv(f, sep="\t")
        print(f, len(x), "rows")
    except FileNotFoundError:
        print(f, "not found (0 rows)")

dropped: 4444 | positive rate: 0.218
GeneSymbol
TTN      331
MECP2    185
OBSCN    113
MACF1    105
DYSF      99
ARSA      88
RUNX1     77
KCNT1     68
DST       61
PLEC      61
Name: count, dtype: int64
missing_seq.tsv 13 rows
ref_fail_windowed.tsv 4198 rows


### Five-fold evaluation on the windowed scores

In [26]:
d = pd.read_csv("scored_windowed.tsv", sep="\t")
emb = np.load("esm_emb_w.npy").astype(np.float32)
zs = np.load("esm_zs_w.npy")
y = d.label.values
groups = d.GeneSymbol.values
win = d.windowed.values

B = substitution_matrices.load("BLOSUM62")
blosum = np.array([B[w][m] for w, m in zip(d.wt, d.mut)])

subsets = {"all": np.ones(len(d), bool), "full-length": win == 0, "windowed": win == 1}

def metrics(yt, s):
    return roc_auc_score(yt, s), average_precision_score(yt, s)

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
records = []
oof_clf = np.full(len(d), np.nan)

for k, (tr, te) in enumerate(sgkf.split(emb, y, groups)):
    assert set(groups[tr]).isdisjoint(groups[te])
    sc = StandardScaler().fit(emb[tr])
    clf = LogisticRegression(C=0.1, max_iter=2000, class_weight="balanced")
    clf.fit(sc.transform(emb[tr]), y[tr])
    oof_clf[te] = clf.decision_function(sc.transform(emb[te]))

    scores = {"BLOSUM62": -blosum, "ESM-2 zero-shot": -zs, "ESM-2 embedding + logreg": oof_clf}
    for sname, mask in subsets.items():
        t = te[mask[te]]
        if len(np.unique(y[t])) < 2:
            continue
        for mname, s in scores.items():
            auc, ap = metrics(y[t], s[t])
            records.append(dict(fold=k, subset=sname, method=mname,
                                n=len(t), pos_rate=y[t].mean(),
                                AUROC=auc, PRAUC=ap))

res = pd.DataFrame(records)
summary = (res.groupby(["subset", "method"])
              .agg(n=("n", "mean"), pos_rate=("pos_rate", "mean"),
                   AUROC_mean=("AUROC", "mean"), AUROC_sd=("AUROC", "std"),
                   PRAUC_mean=("PRAUC", "mean"), PRAUC_sd=("PRAUC", "std"))
              .round(3))
print(summary.to_string())
res.to_csv("cv_results_windowed.tsv", sep="\t", index=False)

                                            n  pos_rate  AUROC_mean  AUROC_sd  PRAUC_mean  PRAUC_sd
subset      method                                                                                 
all         BLOSUM62                  12380.2     0.390       0.664     0.008       0.516     0.042
            ESM-2 embedding + logreg  12380.2     0.390       0.898     0.010       0.868     0.022
            ESM-2 zero-shot           12380.2     0.390       0.903     0.010       0.881     0.019
full-length BLOSUM62                   6808.6     0.476       0.654     0.005       0.595     0.039
            ESM-2 embedding + logreg   6808.6     0.476       0.885     0.009       0.877     0.018
            ESM-2 zero-shot            6808.6     0.476       0.893     0.010       0.892     0.013
windowed    BLOSUM62                   5571.6     0.272       0.683     0.012       0.396     0.117
            ESM-2 embedding + logreg   5571.6     0.272       0.906     0.014       0.829     0.083


### Dropped set = 4444 | accounted for = 4198 | missing_seq = 13 | unexplained = 233 ... why??

In [28]:
p = pd.read_csv("parsed_missense.tsv", sep="\t")
s = pd.read_csv("scored_windowed.tsv", sep="\t")
r = pd.read_csv("ref_fail_windowed.tsv", sep="\t")
m = pd.read_csv("missing_seq.tsv", sep="\t")
seqs = json.load(open("uniprot_seqs.json"))

for f in ["scored_windowed.tsv", "ref_fail_windowed.tsv", "missing_seq.tsv", "uniprot_seqs.json"]:
    print(f"{f:25s} modified {time.ctime(os.path.getmtime(f))}")

explained = set(r.VariationID) | set(m.VariationID)
dropped = p[~p.VariationID.isin(s.VariationID) & ~p.VariationID.isin(explained)].copy()
dropped["seq"] = dropped.GeneSymbol.map(seqs)
dropped["cache_len"] = dropped.seq.map(lambda x: len(x) if isinstance(x, str) else 0)
dropped["wt_full"] = [
    (x[q - 1] == w) if isinstance(x, str) and 0 < q <= len(x) else None
    for x, q, w in zip(dropped.seq, dropped.pos, dropped.wt)
]

print("\nunexplained:", len(dropped))
print("no cached sequence:", (dropped.cache_len == 0).sum())
print("position beyond sequence end:", ((dropped.cache_len > 0) & dropped.wt_full.isna()).sum())
print("wild-type matches full sequence:", (dropped.wt_full == True).sum())
print("wild-type mismatch on full sequence:", (dropped.wt_full == False).sum())
print("\ngenes with cache length 0:")
print(dropped.loc[dropped.cache_len == 0, "GeneSymbol"].value_counts().head(10))

scored_windowed.tsv       modified Fri Oct 23 16:04:08 2026
ref_fail_windowed.tsv     modified Fri Oct 23 15:08:00 2026
missing_seq.tsv           modified Fri Oct 23 15:07:58 2026
uniprot_seqs.json         modified Fri Oct 23 15:01:12 2026

unexplained: 233
no cached sequence: 0
position beyond sequence end: 233
wild-type matches full sequence: 0
wild-type mismatch on full sequence: 0

genes with cache length 0:
Series([], Name: count, dtype: int64)


### Write the missing 233 sequences to a new file

In [29]:
p = pd.read_csv("parsed_missense.tsv", sep="\t")
s = pd.read_csv("scored_windowed.tsv", sep="\t")
r = pd.read_csv("ref_fail_windowed.tsv", sep="\t")
m = pd.read_csv("missing_seq.tsv", sep="\t")
seqs = json.load(open("uniprot_seqs.json"))

explained = set(s.VariationID) | set(r.VariationID) | set(m.VariationID)
rest = p[~p.VariationID.isin(explained)].copy()
rest["cache_len"] = rest.GeneSymbol.map(lambda g: len(seqs.get(g) or ""))
rest["reason"] = "pos_out_of_range"
rest.to_csv("pos_out_of_range.tsv", sep="\t", index=False)
print(len(rest), "variants written to pos_out_of_range.tsv")
print(rest.GeneSymbol.nunique(), "genes")

233 variants written to pos_out_of_range.tsv
99 genes
